<a href="https://colab.research.google.com/github/charlyacha/labo1-colabs/blob/main/02_Mediciones_indirectas_y_propagacion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# Colab 02 — Mediciones indirectas, propagación de incertezas y cómo se reporta un resultado

**Laboratorio 1 · Clase 2**

**Objetivos.**

1. Distinguir una medición **directa** de una **indirecta** (O2.1).
2. Propagar incertezas con la fórmula general, sin memorizar casos particulares, y enunciar sus dos
   hipótesis (O2.2).
3. Derivar con `sympy` y evaluar con `lambdify` (O2.3).
4. Leer la **tabla de contribuciones** y decidir con ella qué medición mejorar primero (O2.4).
5. Escribir un resultado con el **redondeo correcto** del par (valor, incerteza) (O2.5).
6. Reconocer el error de cero y la calibración como sistemáticos, y distinguir exactitud de
   precisión (O2.6).

**Requisitos previos:** Colab 01.

> **Antes de empezar:** hacé `Archivo → Guardar una copia en Drive`.

In [ ]:
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

---
## 1. Directa e indirecta

La longitud se **mide**. El volumen se **calcula**. Nadie tiene un instrumento que lea directamente
el volumen de un cilindro.

Toda magnitud obtenida a partir de otras medidas es una **medición indirecta**, y su incerteza no se
mide: se **propaga**. Ésa es la operación de hoy, y es la primera vez en el curso en que un cálculo
estadístico va a cambiar una decisión experimental.

Un detalle sobre el que conviene ser explícito antes de seguir: cuando medís el diámetro en varias
posiciones del cilindro y los números difieren, eso puede ser error de medición **o** puede ser que
la pieza no sea cilíndrica. No son lo mismo, no se corrigen igual, y confundirlos es una de las
maneras más rápidas de subestimar una incerteza.

---
## 2. La fórmula general

Si $f$ depende de magnitudes medidas $x_1, \dots, x_n$ con incertezas $\sigma_1, \dots, \sigma_n$
**independientes**, entonces

$$ \sigma_f^2 = \sum_{i=1}^{n} \left(\frac{\partial f}{\partial x_i}\right)^2 \sigma_i^2 $$

Todos los casos que aparecen en los libros (suma, producto, potencia) salen de acá. No hace falta
memorizarlos; conviene entender esta expresión y saber derivar.

**Las dos hipótesis, dichas en voz alta:**

1. **Linealidad local.** Es un desarrollo a primer orden: vale si las incertezas son chicas frente a
   los valores. Si $\sigma_x / x$ es del 30 %, la fórmula miente.
2. **Independencia.** Si medís $D$ y $h$ con el **mismo** calibre descalibrado, los errores están
   correlacionados y hay un término cruzado $2\,\partial_D f\,\partial_h f\,\mathrm{cov}(D,h)$ que
   esta fórmula ignora.

Dos observaciones que suelen pasarse por alto:

- Los términos se suman **en cuadratura**. Una contribución que sea un tercio de otra aporta un
  noveno a la varianza: es decir, casi nada.
- Para productos y potencias, la fórmula se reduce a sumar en cuadratura los errores **relativos**,
  cada uno pesado por su exponente. Lo vamos a verificar simbólicamente en el Ejercicio 2.4.

> ### Una deuda que se declara acá y se paga en el Colab 03
>
> Todo lo que sigue trata la **apreciación del instrumento como si fuera un $\sigma$**. Eso no es
> obvio y no vamos a hacer de cuenta que sí.
>
> La justificación ya la viste en el Colab 01, Sección 3: el error de resolución se distribuye
> **uniformemente** en un intervalo de ancho $\Delta$, y una distribución uniforme tiene desviación
> estándar $\sigma = \Delta/\sqrt{12}$. Ése es el $\sigma$ que corresponde.
>
> En la práctica, mucha gente usa directamente $\Delta$ o $\Delta/2$ como incerteza, que es más
> conservador (sobreestima el error en un factor 3,5 o 1,7). No está mal, pero **hay que saber cuál
> se está usando y declararlo**, porque cambia el resultado de todo test de compatibilidad
> posterior. En este curso usamos $\Delta/\sqrt{12}$ cuando la resolución es la única fuente, y la
> dispersión medida cuando la hay.

---
## 3. Derivadas simbólicas con SymPy

Derivar a mano una expresión con cinco variables es una fuente de errores gratuita. SymPy lo hace
exacto. Igual conviene verificar **una** derivada a mano, para que la caja no arranque negra.

In [ ]:
# Densidad de un cilindro, escrita con el DIÁMETRO, que es lo que se mide:
#     rho = 4 m / (pi * D^2 * h)
m, D, h = sp.symbols('m D h', positive=True)
rho = 4*m / (sp.pi * D**2 * h)

print("ρ =", rho)
print()
for var in (m, D, h):
    print(f"∂ρ/∂{var} =", sp.simplify(sp.diff(rho, var)))

Fijate en $\partial\rho/\partial D = -8m/(\pi D^3 h) = -2\rho/D$. El factor 2 es el exponente del
diámetro, y es la razón por la que el diámetro va a dominar la incerteza aunque esté **mejor medido**
que la masa. Verificá esta derivada a mano antes de seguir.

Y una decisión que parece cosmética y no lo es: escribimos la fórmula en función de $D$ y no de $r$
porque **$D$ es lo que mide el calibre**. Si escribís $\rho = m/(\pi r^2 h)$ y después ponés
$r = D/2$ con $\sigma_r = \sigma_D/2$, llegás al mismo lado; pero cada paso intermedio es una
oportunidad de perder un factor 2. La regla general: propagá sobre las variables que efectivamente
mediste.

---
## 4. Propagar y leer la tabla de contribuciones

In [ ]:
def propagar(expr, variables, valores, errores, nombre='f'):
    '''Propaga incertezas de una expresión simbólica e imprime la tabla de contribuciones.

    expr      : expresión de SymPy
    variables : lista de símbolos
    valores   : dict {símbolo: valor medido}
    errores   : dict {símbolo: incerteza}
    Devuelve (valor, incerteza).
    '''
    valor = float(expr.subs(valores))
    contribuciones = {}
    for v in variables:
        d = float(sp.diff(expr, v).subs(valores))
        contribuciones[v] = (d * errores[v])**2
    var_total = sum(contribuciones.values())
    sigma = np.sqrt(var_total)

    print(f"{nombre} = {valor:.6g}  ±  {sigma:.3g}      "
          f"(error relativo: {100*sigma/abs(valor):.2f} %)")
    print()
    print(f"{'variable':>10s} {'valor':>12s} {'σ':>12s} {'σ rel.':>9s} "
          f"{'contrib. a σ²':>15s}")
    print("-" * 64)
    for v in variables:
        frac = contribuciones[v] / var_total
        print(f"{str(v):>10s} {float(valores[v]):12.6g} {errores[v]:12.3g} "
              f"{100*errores[v]/float(valores[v]):8.2f}% {100*frac:14.1f}%")
    return valor, sigma

In [ ]:
# Un caso concreto: cilindro metálico medido con calibre (0,05 mm) y balanza (0,01 g)
#   - σ_m : resolución de la balanza -> 1e-5 kg (dominada por la resolución)
#   - σ_D : dispersión entre posiciones -> 0,03 mm  (MAYOR que la resolución: la pieza no es
#           perfectamente cilíndrica, y ésa es la incerteza que corresponde)
#   - σ_h : ídem, 0,03 mm
valores = {m: 0.16745, D: 0.02500, h: 0.04020}      # kg, m, m
errores = {m: 0.00001, D: 0.00003, h: 0.00003}      # kg, m, m

rho_val, rho_err = propagar(rho, [m, D, h], valores, errores, nombre='ρ [kg/m³]')

**Leé la tabla, que es el resultado útil.** No dice cuánto vale la densidad: dice **dónde invertir
esfuerzo experimental**. El diámetro aporta más del 90 % de la varianza, y no porque esté peor
medido —su error relativo es intermedio— sino porque entra al cuadrado.

Conclusión operativa: conseguir una balanza mejor no cambia **nada** (la masa aporta menos del 1 %
de la varianza, y bajarle el error a la mitad mejoraría el resultado en la cuarta cifra). Medir mejor
el diámetro, sí.

Este razonamiento es exactamente el que se usa al diseñar un experimento de investigación, y es lo
que se le va a pedir en la Práctica Especial: estimar *a priori* qué error esperás y de dónde viene.

> **Ejercicio 2.1.** Bajá a la mitad la incerteza de la variable dominante y volvé a correr la celda.
> ¿Cuánto mejoró el resultado? Ahora bajá a la mitad la de la variable que menos aporta. Compará el
> esfuerzo experimental de cada una contra su beneficio.

---
## 5. Del símbolo al número, en cantidad: `lambdify`

Cuando tenés que propagar sobre muchos puntos (por ejemplo, una columna entera de datos), evaluar
con `subs()` es lentísimo. `lambdify` convierte la expresión simbólica en una función de NumPy.

In [ ]:
sig_m, sig_D, sig_h = sp.symbols('sigma_m sigma_D sigma_h', positive=True)
var_rho = ((sp.diff(rho, m))**2 * sig_m**2 +
           (sp.diff(rho, D))**2 * sig_D**2 +
           (sp.diff(rho, h))**2 * sig_h**2)
sigma_rho = sp.sqrt(var_rho)

f_rho   = sp.lambdify((m, D, h), rho, 'numpy')
f_sigma = sp.lambdify((m, D, h, sig_m, sig_D, sig_h), sigma_rho, 'numpy')

# ahora sobre arreglos completos: cuatro cilindros del mismo material
masas     = np.array([0.16745, 0.16702, 0.16788, 0.16731])
diametros = np.array([0.02500, 0.02498, 0.02503, 0.02499])
alturas   = np.array([0.04020, 0.04015, 0.04024, 0.04018])

rhos   = f_rho(masas, diametros, alturas)
sigmas = f_sigma(masas, diametros, alturas, 0.00001, 0.00003, 0.00003)

for i, (v, s) in enumerate(zip(rhos, sigmas), 1):
    print(f"cilindro {i}:  ρ = {v:8.1f} ± {s:5.1f} kg/m³")

---
## 6. Cómo se escribe el resultado

Esto no es una convención tipográfica: es la primera cosa que se corrige en un informe. Y es acá y no
en el Colab 01 porque acá aparece el primer resultado **calculado** del curso.

**Regla operativa:**

1. La incerteza se redondea a **una cifra significativa**. Excepción usual: si esa cifra es 1 o 2 se
   conservan dos, porque redondear 0,14 a 0,1 pierde casi un 30 % de la información.
2. El valor se redondea **a la misma posición decimal** que la incerteza.
3. Se escriben ambos con la misma unidad.

Escribir `ρ = 8485,73218 ± 20 kg/m³` es afirmar que conocés la séptima cifra de un número cuya
cuarta ya es dudosa. Es una contradicción interna, y se corrige en rojo.

In [ ]:
from math import floor, log10

def redondear_con_error(x, dx, dos_cifras_si_empieza_en_1_o_2=True):
    '''Devuelve (valor, error, n_decimales) redondeados según la regla estándar.'''
    if dx <= 0:
        raise ValueError("La incerteza debe ser positiva.")
    orden   = floor(log10(abs(dx)))
    primera = int(dx / 10**orden)
    cifras  = 2 if (dos_cifras_si_empieza_en_1_o_2 and primera in (1, 2)) else 1
    dec     = -(orden - (cifras - 1))
    return round(x, dec), round(dx, dec), max(dec, 0)

def reportar(x, dx, unidad=""):
    '''Devuelve el string '(valor ± error) unidad' con las cifras correctas.'''
    x_r, dx_r, dec = redondear_con_error(x, dx)
    return f"({x_r:.{dec}f} ± {dx_r:.{dec}f}) {unidad}".strip()

# Casos de prueba: mirá qué hace en cada uno
for x, dx, u in [(rho_val, rho_err, "kg/m³"),
                 (9.8123, 0.0456, "m/s²"),
                 (9.8123, 0.0156, "m/s²"),      # el error empieza en 1 -> dos cifras
                 (1234.5678, 12.0, "s"),
                 (0.00023456, 0.0000031, "m")]:
    print(f"{x:>12.6g} ± {dx:<9.4g} ->  {reportar(x, dx, u)}")

> **Ejercicio 2.2.** El tercer caso conserva dos cifras en la incerteza y el segundo no, aunque los
> dos valores son casi iguales. Escribí en una oración cuál es la regla y por qué existe.
> A partir de ahora, **toda** entrega usa `reportar()`.

---
## 7. Contestar el título: ¿de qué material está hecho?

Con el resultado y su incerteza, la comparación con una tabla ya no es "se parece a". Es un test.

In [ ]:
# Densidades de referencia, kg/m³. Ojo: las aleaciones (latón, bronce) no tienen un valor único
# sino un rango, según la composición. Eso también es parte de la respuesta honesta.
tabla = {"aluminio": 2700, "titanio": 4500, "zinc": 7135, "hierro": 7874,
         "latón": 8500, "cobre": 8960, "plomo": 11340}

def candidatos(valor, sigma, tabla, z_max=3.0):
    print(f"resultado: {reportar(valor, sigma, 'kg/m³')}\n")
    print(f"{'material':>12s} {'ρ tabla':>10s} {'z':>8s}   veredicto")
    print("-" * 52)
    for nombre, ref in sorted(tabla.items(), key=lambda kv: kv[1]):
        z = abs(valor - ref) / sigma
        marca = "COMPATIBLE" if z < z_max else ""
        print(f"{nombre:>12s} {ref:10d} {z:8.1f}   {marca}")

candidatos(rho_val, rho_err, tabla)

> **Ejercicio 2.3.** Rehacé la propagación suponiendo que mediste todo con **regla** en vez de
> calibre. La resolución es 1 mm, así que $\sigma_D = \sigma_h = 1\,\text{mm}/\sqrt{12} = 0{,}29$ mm.
> Volvé a correr `candidatos()`. ¿Cuántos
> materiales sobreviven ahora? La respuesta honesta pasa a ser "es uno de estos dos", y eso **no es
> un fracaso**: es el resultado que tus datos soportan. Un resultado que afirma más de lo que los
> datos permiten está mal aunque acierte.

---
## 8. Sistemáticos instrumentales

El error de cero que mediste en la Clase 1 es un sistemático concreto que ya tenés anotado. Tiene dos
propiedades que lo separan del error aleatorio:

- **No se reduce repitiendo.** Está en todas las mediciones por igual.
- **Se corrige restándolo**, si lo mediste. Si no lo mediste, no lo podés corregir y tampoco lo vas a
  ver en la dispersión.

De ahí sale la distinción entre **precisión** (dispersión chica) y **exactitud** (cercanía al valor
verdadero). Son independientes: se puede ser muy preciso y muy inexacto, y ése es el caso peligroso,
porque el resultado *se ve* bien.

In [ ]:
# Un calibre descalibrado en +0,10 mm midiendo un cilindro de 12,50 mm de diámetro
rng = np.random.default_rng(20260819)
D_verdadero = 12.500                                   # mm
lecturas = rng.normal(D_verdadero + 0.10, 0.012, 20)   # muy preciso, sesgado

media = lecturas.mean()
sem   = lecturas.std(ddof=1) / np.sqrt(len(lecturas))

print(f"D verdadero : {D_verdadero:.3f} mm")
print(f"D medido    : {reportar(media, sem, 'mm')}")
print(f"dispersión  : {lecturas.std(ddof=1):.4f} mm   <- excelente precisión")
print(f"discrepancia: {abs(media - D_verdadero)/sem:.1f} σ  <- pésima exactitud")

Veinte mediciones muy consistentes entre sí, y el resultado está a decenas de sigmas del valor
verdadero. La dispersión **no lo detecta y no puede detectarlo**: el sesgo está en todas las
mediciones por igual.

La contracara estadística de esto —qué pasa cuando aumentás $N$ con un sistemático presente— se ve en
el Colab 03 con tus datos de tiempo de reacción, y es peor de lo que parece.

---
## 9. Ejercicios

**2.4.** Verificá **simbólicamente** con SymPy que para $f = x/y$ la propagación se reduce a
$(\sigma_f/f)^2 = (\sigma_x/x)^2 + (\sigma_y/y)^2$. Repetilo para $f = x^n$ y mostrá que
$\sigma_f/f = |n|\,\sigma_x/x$.

**2.5.** Propagá la incerteza de $g$ en la medición con péndulo, $g = 4\pi^2 L/T^2$, con
$L = (1{,}000 \pm 0{,}002)$ m y $T = (2{,}006 \pm 0{,}004)$ s. ¿Qué variable domina? ¿Cuál conviene
medir mejor? Guardá la respuesta: la vas a necesitar en la Clase 4, donde vas a medir $T$ en serio.

**2.6.** Tu grupo mide el diámetro de un cilindro en cinco posiciones distintas y obtiene una
dispersión de 0,04 mm con un calibre de 0,05 mm de resolución. ¿Esa dispersión es error de medición o
falta de cilindricidad? Proponé una medición que permita distinguir los dos casos.

**2.7.** *(sobre la hipótesis de independencia)* Medís $D$ y $h$ del cilindro con el mismo calibre,
que tiene un error de cero de $+\varepsilon$ desconocido. Escribí $V(D-\varepsilon, h-\varepsilon)$ y
desarrollá a primer orden en $\varepsilon$. ¿El efecto sobre $V$ se cancela, se suma, o depende de la
geometría? ¿Qué pasa con la fórmula de propagación estándar en este caso?

**2.8.** Con tus datos reales de la Clase 2, completá la Entrega corta 1: densidad con propagación
completa, tabla de contribuciones, resultado con `reportar()`, y una frase sobre qué medición
mejorarías primero y por qué.